# OghmaNano Bragg grating (FDTD) — 正向 I_new + Emission TMM 对齐

Oghma v8.1 FDTD 项目 [`01_hello_bragg_grating`](../../oghma_projects/bragg_grating/01_hello_bragg_grating)：
用 [`00_software_alignment_skills.md`](00_software_alignment_skills.md) 正向构造 `I_new(λ)` 驱动 **emission TMM**（hs 偶极子 @ `z_source`，`u=0`），预测传递与耦合输出谱。

本 notebook 处理 detector0 混源+反射问题，并用 **Case A / B / C** 三条路径（emission 传递比、emission 耦合谱、被动 T 包络）验证互易性。

**运行前提**（[README.md](../../../../README.md)）：`build/simulation.so`；venv + `LD_LIBRARY_PATH`。


In [ ]:
import numpy as np

from oghma_runtime import bootstrap_tmm_session

REPO, BUILD, TMM_DIR = bootstrap_tmm_session()
import simulation

from oghma_bragg import (
    bragg_fdtd_stack_labels,
    bragg_fdtd_stack_thicknesses_um,
    build_bragg_fdtd_stack,
    compute_bragg_emission_case_bc_spectra,
    compute_bragg_emission_transfer,
    compute_bragg_lam_e_norm,
    compute_bragg_reflection_ahead_of_z,
    parse_bragg_grating_geometry,
    plot_bragg_fdtd_stack_section,
)
from oghma_core import compare_metrics, fdtd_input_combine_power
from oghma_fdtd_alignment import (
    display_alignment_reports,
    evaluate_case,
    load_fdtd_alignment_bundle,
    plot_case_summary_grid,
    s_in_times_intensity,
)
from oghma_fdtd_source import predict_fdtd_output_spectrum_incoherent

bundle = load_fdtd_alignment_bundle(
    REPO,
    "oghma_projects/bragg_grating/01_hello_bragg_grating",
    parse_bragg_grating_geometry,
)
MASK_BRAGG = (bundle.wl_um >= 0.43) & (bundle.wl_um <= 0.72)
MASK_PASSBAND = (~MASK_BRAGG) | (bundle.s_out > 0.3 * np.max(bundle.s_out))

print(f"BUILD={BUILD}")
print(f"periods={bundle.geom.n_periods}, Λ={bundle.geom.period_um * 1e3:.1f} nm")
print(f"z_source={bundle.layout.z_source_um * 1e3:.0f} nm, z_det_in={bundle.layout.z_detector_in_um * 1e3:.0f} nm")
print(f"λ: {bundle.wl_um[0]*1e3:.0f}–{bundle.wl_um[-1]*1e3:.0f} nm ({len(bundle.wl_um)} pts)")
print(f"FDTD source: {bundle.fdtd_src.waveform}, Ey={bundle.fdtd_src.excite_ey}")
print(f"emission dipole: orient=hs, u=0, z={bundle.layout.z_source_um * 1e3:.0f} nm")


## §1 FDTD 膜系与 emission 偶极子

膜系同被动 notebook；emission 偶极子 @ `z_source`，Ey → `hs`，`u=0`（1D 法向）。

In [ ]:
layers_demo = build_bragg_fdtd_stack(
    bundle.geom, bundle.layout, 500.0, simulation
)
labels = bragg_fdtd_stack_labels(bundle.geom, bundle.layout)
thicknesses_nm = bragg_fdtd_stack_thicknesses_um(bundle.geom, bundle.layout)
plot_bragg_fdtd_stack_section(
    bundle.geom,
    bundle.layout,
    simulation,
)
print(f"emission dipole: orient=hs, u=0, z={bundle.layout.z_source_um * 1e3:.0f} nm")


## §2 等效光源(消除时间项)
预期结果 : `S_in = I_new × G_in`  
TODO : 分析原因


In [ ]:
import numpy as np
r_det = compute_bragg_reflection_ahead_of_z(
    bundle.layout.z_detector_in_um * 1e3,
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    )
g_in = fdtd_input_combine_power(r_det)
s_pred_det0 = predict_fdtd_output_spectrum_incoherent(bundle.i_new, g_in)
s_in_norm = s_in_times_intensity(bundle)

evaluate_case(
    bundle,
    "§2 I_new vs S_IN×Intensity",
    bundle.i_new,
    s_in_norm,
    mask=MASK_BRAGG,
    mask_label="430,720",
    ylabel="|E|²",
    title="§2: I_new vs S_IN×Intensity",
    tmm_label="I_new",
    baseline_name="S_IN×Int",
)
# evaluate_case(
#     bundle,
#     "§2 I_new×G_in vs S_IN",
#     s_pred_det0,
#     bundle.s_in,
#     mask=MASK_BRAGG,
#     mask_label="430,720",
#     ylabel="|E|²",
#     title="§2: I_new×G_in vs detector0",
#     tmm_label="I_new×G_in",
#     baseline_name="S_IN (det0)",
# )


## §3 Case A — 传递比 T(λ)

$T/|1+r|^2$ = $S_{out}/S_{in}$

In [ ]:
import numpy as np
em = compute_bragg_emission_transfer(
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    simulation,
    z_um=bundle.layout.z_source_um,
    u=0.0,
    orient="hs",
)
T_EM = em["t_trans_ratio_fdtd"]
T_PASSIVE = compute_bragg_lam_e_norm(
    bundle.geom, bundle.layout, bundle.wl_um, simulation
)

evaluate_case(
    bundle,
    "Case A T_em",
    T_EM,
    T_PASSIVE,
    mask=MASK_BRAGG,
    mask_label="430,720",
    ylabel="T/G_in",
    title="Case A: emission T/G_in vs T_passive/G_in",
    tmm_label="T_em",
    baseline_name="T_passive/G_in",
)


## §4 Case B / C — 耦合输出谱 vs `detector1/lam_E`

**验收基准**：`S_OUT`（FDTD `detector1`）。

- **Case B**：相干 $\mathrm{flux}(E_{bot}^{sum})$，源谱用正向 $I_{new}(\lambda)$
- **Case C**：$T_{passive}(\lambda)\, I_{new}(\lambda)$（[`00_software_alignment_skills.md`](00_software_alignment_skills.md) 非相干输出；不用 $S_{IN}$，因其混入了源与反射）


In [ ]:
import numpy as np
case_bc = compute_bragg_emission_case_bc_spectra(
    simulation,
    bundle.geom,
    bundle.layout,
    bundle.wl_um,
    bundle.s_in,
    spectrum=bundle.i_new,
    t_passive=T_PASSIVE,
)
S_PRED_B = case_bc["s_pred_b"]
S_PRED_C = case_bc["s_pred_c"]

for tag, mask in [
    ("full", np.ones_like(bundle.wl_um, dtype=bool)),
    ("Bragg [430,720]", MASK_BRAGG),
    ("passband / far stop", MASK_PASSBAND),
]:
    sb = compare_metrics(S_PRED_B[mask], bundle.s_out[mask], x=bundle.wl_um[mask])
    sc = compare_metrics(S_PRED_C[mask], bundle.s_out[mask], x=bundle.wl_um[mask])
    print(f"[{tag}] B corr={sb['corr']:.4f}  C corr={sc['corr']:.4f}")

evaluate_case(
    bundle,
    "Case B P_bot",
    S_PRED_B,
    bundle.s_out,
    mask=MASK_BRAGG,
    mask_label="430,720",
    ylabel="|E|²",
    title="Case B: coherent flux(E_bot) vs S_OUT",
    tmm_label="flux(E_bot_sum)",
    baseline_name="S_OUT",
)
evaluate_case(
    bundle,
    "Case C T_passive·I_new",
    S_PRED_C,
    bundle.s_out,
    mask=MASK_BRAGG,
    mask_label="430,720",
    ylabel="|E|²",
    title="Case C: T_passive·I_new vs S_OUT",
    tmm_label="T_passive·I_new",
    baseline_name="S_OUT",
)
evaluate_case(
    bundle,
    "Case D emission vs passive",
    S_PRED_B,
    S_PRED_C,
    mask=MASK_BRAGG,
    mask_label="430,720",
    ylabel="|E|²",
    title="Case D: emission vs passive",
    tmm_label="Case B",
    baseline_name="Case C",
)


## §6 Case 汇总

汇总 `CASE_SUMMARY`，并同时绘制 raw 与 peak-normalized 对比图。


In [ ]:
plot_case_summary_grid(
    bundle,
    [
        {
            "title": "Case A",
            "pred": T_EM,
            "baseline": T_PASSIVE,
            "pred_label": "T_em",
            "baseline_label": "T_passive/G_in",
            "ylabel": "T/G_in",
        },
        {
            "title": "Case B",
            "pred": S_PRED_B,
            "baseline": bundle.s_out,
            "pred_label": "Case B",
            "baseline_label": "FDTD S_out",
        },
        {
            "title": "Case C",
            "pred": S_PRED_C,
            "baseline": bundle.s_out,
            "pred_label": "Case C",
            "baseline_label": "FDTD S_out",
        },
        {
            "title": "Case D",
            "pred": S_PRED_B,
            "baseline": S_PRED_C,
            "pred_label": "Case B (emission flux)",
            "baseline_label": "Case C (T_passive·I_new)",
        },
    ],
)


## §7 对齐报告


In [ ]:
display_alignment_reports(bundle)

print("\n--- FDTD → TMM 映射要点 ---")
print("1. Case B/C baseline = detector1 lam_E (S_OUT); §2 S_source is provisional")
print("2. FDTD Ey → emission hs dipole, u=0")
print(f"3. dipole z = {bundle.layout.z_source_um * 1e3:.0f} nm; stack = build_bragg_fdtd_stack")
print("4. evaluate_case(bundle, ...): raw + peak-normalized compare/plots")
for k, v in bundle.case_summary.items():
    line = f"   {k}: corr={v['corr']:.4f}, RMSE={v['rmse']:.4f}"
    line += f", norm-corr={v.get('norm_corr', float('nan')):.4f}"
    print(line)
print("5. PML → semi-infinite air; pre/post air from FDTD z geometry")


## §8 总结

**Case B/C 以 `S_OUT` 对齐结果为据**（非草稿中的固定 corr 值）。

| Case | 路径 | 公式 | Baseline |
|---|---|---|---|
| **A** | 忽略时间相干效应（如果这个效应存在的话...） | $S_{in} = I_{new} × |1+r|^2$ | `detector0` |
| **B** | emission 相干 flux | $\mathrm{flux}(E_{bot}^{sum})$, 源谱 $I_{new}(\lambda)$ | `detector1` |
| **C** | 被动 TMM | $T_{passive}\cdot I_{new}$ | `detector1` |

Case A 我的预期是包络相同， 这里的 '时间相干性' 应该怎么理解呢?   
Case B/C 均用等效源谱 $I_{new}$。  
Case C 为被动非相干包络，Bragg 窗内 corr 通常与 Case B 同量级。阻带峰位应对齐 ~505 nm。  
detector 能量和 tmm 我怀疑量纲没有对齐，导致只有归一化之后形状才能接近。  
